# Streaming Simulation — Fraud Detection Pipeline

This notebook simulates real-time streaming by:
1. Splitting the enriched parquet into 1000-row batches
2. Setting up Spark Structured Streaming to read those batches
3. Flagging suspicious transactions per micro-batch
4. Writing results to `/processed/stream_output`

In [1]:
import os
import shutil
import time

# ─── Workaround: fake HADOOP_HOME para evitar UnsatisfiedLinkError en Windows ─
os.environ.setdefault("HADOOP_HOME", "C:/hadoop")
os.makedirs("C:/hadoop/bin", exist_ok=True)

# ─── Paths ────────────────────────────────────────────────────────────────────
PROCESSED_DIR  = "D:/UP/BigData/Pryecto1/processed"
ENRICHED_PATH  = os.path.join(PROCESSED_DIR, "transactions_enriched")
STREAM_INPUT   = os.path.join(PROCESSED_DIR, "stream_input")
STREAM_OUTPUT  = os.path.join(PROCESSED_DIR, "stream_output")
CHECKPOINT_DIR = os.path.join(PROCESSED_DIR, "stream_checkpoint")

print(f"Enriched path  : {ENRICHED_PATH}")
print(f"Stream input   : {STREAM_INPUT}")
print(f"Stream output  : {STREAM_OUTPUT}")


Enriched path  : D:/UP/BigData/Pryecto1/processed\transactions_enriched
Stream input   : D:/UP/BigData/Pryecto1/processed\stream_input
Stream output  : D:/UP/BigData/Pryecto1/processed\stream_output


## 1 — Start Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

# Mismo setup que batch_processing ─────────────────────────────────────────────
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

print("Starting Spark session ...")

spark = (
    SparkSession.builder
    .appName("FraudDetection_StreamingSimulation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")          # era 2g — insuficiente
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.schemaInference", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark session started — version {spark.version}")


Starting Spark session ...
✓ Spark session started — version 4.1.1


## 2 — Split Enriched Data into Batches of 1000 Rows

In [3]:
print("=" * 60)
print("STEP 2 — Splitting enriched data into stream batches")
print("=" * 60)

if os.path.exists(STREAM_INPUT):
    shutil.rmtree(STREAM_INPUT)
    print(f"  Cleared previous stream_input directory")
os.makedirs(STREAM_INPUT, exist_ok=True)

print("  Loading enriched parquet ...")
enriched = spark.read.parquet(ENRICHED_PATH)

# Detectar columna timestamp
ts_col = next(
    (c for c in enriched.columns
     if c in ("timestamp", "date", "trans_date", "transaction_date")),
    None
)
if ts_col:
    print(f"  Sorting by timestamp column: {ts_col}")
    enriched = enriched.orderBy(F.col(ts_col).asc_nulls_last())
else:
    print("  No timestamp column found — using natural order")

total_rows = enriched.count()
print(f"  Total rows to batch: {total_rows:,}")

# Escribir directamente con Spark — evita traer 13M filas al driver
BATCH_SIZE = 1000
num_partitions = max(1, total_rows // BATCH_SIZE)
print(f"  Writing {num_partitions:,} batch files via Spark (no toPandas) ...")

enriched.repartition(num_partitions).write.mode("overwrite").parquet(STREAM_INPUT)
batch_count = num_partitions

print(f"  ✓ Created {batch_count:,} batch files in {STREAM_INPUT}")
print(f"  Each batch: ~{BATCH_SIZE} rows")


STEP 2 — Splitting enriched data into stream batches
  Cleared previous stream_input directory
  Loading enriched parquet ...
  Sorting by timestamp column: date
  Total rows to batch: 13,305,915
  Writing 13,305 batch files via Spark (no toPandas) ...
  ✓ Created 13,305 batch files in D:/UP/BigData/Pryecto1/processed\stream_input
  Each batch: ~1000 rows


## 3 — Set Up Spark Structured Streaming

In [4]:
print("=" * 60)
print("STEP 3 — Configuring Structured Streaming source")
print("=" * 60)

# Infer schema from the static enriched data
schema = enriched.schema
print(f"  Schema inferred: {len(schema.fields)} fields")

# Create streaming DataFrame reading from stream_input folder
stream_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)   # one batch file per micro-batch
    .parquet(STREAM_INPUT)
)

print("  ✓ Streaming source configured")
print(f"  isStreaming: {stream_df.isStreaming}")

STEP 3 — Configuring Structured Streaming source
  Schema inferred: 46 fields
  ✓ Streaming source configured
  isStreaming: True


## 4 — Flag Suspicious Transactions per Micro-Batch

In [5]:
print("=" * 60)
print("STEP 4 — Defining suspicion logic and streaming query")
print("=" * 60)

# Detect feature columns (may or may not exist)
dev_col = next((c for c in schema.fieldNames() if c == "amount_deviation"), None)
vel_col = next((c for c in schema.fieldNames() if c == "transaction_velocity"), None)

print(f"  amount_deviation col    : {dev_col}")
print(f"  transaction_velocity col: {vel_col}")

# Build suspicion flag
conditions = []
if dev_col:
    conditions.append(F.col(dev_col) > 3)
if vel_col:
    conditions.append(F.col(vel_col) > 10)

if conditions:
    suspicion_flag = conditions[0]
    for cond in conditions[1:]:
        suspicion_flag = suspicion_flag | cond
else:
    # Fallback: flag high-amount transactions if no feature columns
    amt_col = next((c for c in schema.fieldNames() if c == "amount"), None)
    suspicion_flag = F.col(amt_col) > 1000 if amt_col else F.lit(False)

# Apply flag to streaming DataFrame
suspicious_df = stream_df.filter(suspicion_flag)

print("  ✓ Suspicion filter defined")
print("  Rule: amount_deviation > 3 OR transaction_velocity > 10")

STEP 4 — Defining suspicion logic and streaming query
  amount_deviation col    : amount_deviation
  transaction_velocity col: transaction_velocity
  ✓ Suspicion filter defined
  Rule: amount_deviation > 3 OR transaction_velocity > 10


In [6]:
print("=" * 60)
print("STEP 5 — Starting streaming query")
print("=" * 60)

# Clean up previous outputs
for d in [STREAM_OUTPUT, CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"  Cleared: {d}")

# Batch processing function: prints count per micro-batch
batch_stats = {"batch_id": 0, "total_suspicious": 0}

def process_batch(batch_df, batch_id):
    cnt = batch_df.count()
    batch_stats["total_suspicious"] += cnt
    print(f"  [Batch {batch_id:>4}] Suspicious transactions detected: {cnt:>6,}  |  Running total: {batch_stats['total_suspicious']:,}")
    batch_stats["batch_id"] = batch_id

# Write stream output using foreachBatch to also print counts
query = (
    suspicious_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", STREAM_OUTPUT)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="2 seconds")
    .start()
)

print(f"  ✓ Streaming query started — query ID: {query.id}")
print(f"  Writing suspicious transactions to: {STREAM_OUTPUT}")

STEP 5 — Starting streaming query
  ✓ Streaming query started — query ID: 7c36c49b-0be8-426b-abe1-3ec6735ef151
  Writing suspicious transactions to: D:/UP/BigData/Pryecto1/processed\stream_output


In [7]:
print("=" * 60)
print("STEP 6 — Monitoring streaming progress")
print("=" * 60)

# Let streaming run until all batch files are consumed
# We wait up to 120 seconds to process all batches
import time

MAX_WAIT   = 120   # seconds
CHECK_FREQ = 5     # seconds
elapsed    = 0
last_rows  = 0
idle_count = 0

print(f"  Waiting up to {MAX_WAIT}s for streaming to process all batches ...\n")

while elapsed < MAX_WAIT:
    time.sleep(CHECK_FREQ)
    elapsed += CHECK_FREQ

    progress = query.lastProgress
    if progress:
        num_rows    = progress.get("numInputRows", 0)
        total_rows  = progress.get("processedRowsPerSecond", 0)
        batch_id    = progress.get("batchId", "?")
        print(f"  [{elapsed:>3}s] Batch {batch_id} | inputRows={num_rows} | rowsPerSec={total_rows:.1f}")

        # Stop when idle (no new input) for 3 consecutive checks
        if num_rows == 0:
            idle_count += 1
            if idle_count >= 3:
                print("  No new data for 3 checks — stopping query")
                break
        else:
            idle_count = 0

query.stop()
print(f"\n  ✓ Streaming query stopped")

STEP 6 — Monitoring streaming progress
  Waiting up to 120s for streaming to process all batches ...

  [  5s] Batch 1 | inputRows=999 | rowsPerSec=915.7
  [ 10s] Batch 3 | inputRows=999 | rowsPerSec=829.7
  [ 15s] Batch 6 | inputRows=999 | rowsPerSec=867.2
  [ 20s] Batch 8 | inputRows=999 | rowsPerSec=974.6
  [ 25s] Batch 11 | inputRows=999 | rowsPerSec=871.7
  [ 30s] Batch 13 | inputRows=999 | rowsPerSec=540.3
  [ 35s] Batch 16 | inputRows=999 | rowsPerSec=915.7
  [ 40s] Batch 18 | inputRows=999 | rowsPerSec=935.4
  [ 45s] Batch 21 | inputRows=1000 | rowsPerSec=741.3
  [ 50s] Batch 23 | inputRows=1000 | rowsPerSec=832.6
  [ 55s] Batch 26 | inputRows=1000 | rowsPerSec=994.0
  [ 60s] Batch 28 | inputRows=1000 | rowsPerSec=861.3
  [ 65s] Batch 31 | inputRows=1000 | rowsPerSec=552.2
  [ 70s] Batch 33 | inputRows=1000 | rowsPerSec=941.6
  [ 75s] Batch 36 | inputRows=1000 | rowsPerSec=1028.8
  [ 80s] Batch 38 | inputRows=1000 | rowsPerSec=1050.4
  [ 85s] Batch 41 | inputRows=1000 | rowsPer

In [8]:
print("=" * 60)
print("STEP 7 — Verifying stream output")
print("=" * 60)

try:
    output_df = spark.read.parquet(STREAM_OUTPUT)
    total_suspicious = output_df.count()
    print(f"  ✓ Total suspicious transactions written : {total_suspicious:,}")
    print(f"  Output columns: {output_df.columns}")
    output_df.show(5, truncate=True)
except Exception as e:
    print(f"  ! Could not read stream output: {e}")

print("\n  Streaming simulation complete.")
spark.stop()
print("  ✓ Spark session stopped")

STEP 7 — Verifying stream output
  ✓ Total suspicious transactions written : 691
  Output columns: ['id', 'card_id', 'date', 'client_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_code', 'description', 'is_fraud', 'transaction_velocity', 'amount_deviation', 'is_weekend', 'is_night', 'mcc_fraud_rate', 'card_utilization']
+--------+-------+-------------------+---------+-------+-----------------+-----------+-------------+--------------+-------+----+------+--------------+----------+---------+----------------+-------+---+--------+----------------+--------